# CMP7005 PRAC1 — Beijing Air Quality Analysis

**From Data to Application Development**

This notebook delivers Tasks 1–3 of the assessment:

1. **Task 1** — Data selection, ingestion and merging
2. **Task 2** — Exploratory Data Analysis
   - Data understanding
   - Preprocessing (missing values, duplicates, feature engineering, AQI)
   - Statistical and visual analysis (univariate, bivariate, multivariate, temporal)
3. **Task 3** — Predictive model for next-hour PM2.5

Heavy lifting lives in `src/` so the notebook stays a readable narrative.
Tasks 4 (Streamlit app) and 5 (version control) are delivered alongside
this notebook in `app/` and the GitHub repository.

In [ ]:
import sys
from pathlib import Path

# Ensure the project root is on the import path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")

from src import data_loader, preprocessing, eda, model

---
## Task 1 — Data selection, ingestion, and merging

### Station selection

Following the urban / suburban classification in **Yao et al. (2015)** and
**Xu & Zhang (2020)**, the Beijing monitoring stations split broadly into
central-urban sites (within the 4th Ring Road) and suburban sites
(outside the 5th Ring, often closer to mountainous terrain).

We select two stations from each category to enable a clean
urban-vs-suburban comparison:

| Station | Type | Rationale |
| --- | --- | --- |
| **Dongsi** | Urban | Inner-city (Dongcheng District), high traffic density, classic urban-canyon site |
| **Guanyuan** | Urban | Central-west, mixed residential/commercial, included in most Beijing PM2.5 studies |
| **Changping** | Suburban | ~30 km north of central Beijing, predominantly residential, transitional terrain |
| **Huairou** | Suburban | Far north-east mountainous district, regional background site |

Pairing two central sites with two distant sites lets us probe whether
PM2.5 differences are driven by local emissions (urban traffic, residential
heating) or regional transport from the North China Plain.

### Ingestion and merging

In [ ]:
# Download (first run only) and merge the four selected stations
df_raw = data_loader.build_merged_dataset()
print("Merged dataset shape:", df_raw.shape)
df_raw.head()

In [ ]:
# Sanity checks: rows per station, date coverage
print("Rows per station:")
print(df_raw["station"].value_counts())
print("\nDate range per station:")
print(df_raw.groupby("station")["datetime"].agg(["min", "max"]))

Each station contributes 35,064 hourly observations (the 4-year span of
1 March 2013 – 28 February 2017), giving ~140k rows total.

---
## Task 2a — Data understanding

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe(include="all").T

In [ ]:
# Missing values
preprocessing.missing_value_summary(df_raw)

**Initial observations**

* 14 substantive columns plus `datetime` and `station_type`.
* All pollutants and meteorology are stored as floats; `wd` (wind direction)
  is the only categorical numeric.
* Missing values exist across all measured variables — typical for sensor
  networks where instruments occasionally fail or undergo maintenance.
* PM2.5 ranges from a few µg/m³ (clean days) to several hundred (severe
  pollution episodes), with a heavily right-skewed distribution.

---
## Task 2b — Preprocessing

The preprocessing pipeline (`src/preprocessing.preprocess`) does the following:

1. **Duplicate removal** — drop exact duplicates and any duplicate `(station, datetime)` keys.
2. **Missing-value imputation** — within each station, forward-fill then
   back-fill (preserving local autocorrelation), with a station-median fallback.
3. **Datetime feature engineering** — year, month, day, hour, day-of-week,
   weekend flag, season, part-of-day.
4. **AQI computation** — China AQI per HJ 633-2012 (rolling-hourly
   simplification) plus the primary pollutant and a categorical AQI level.

In [ ]:
df = preprocessing.preprocess(df_raw)
print("After preprocessing:", df.shape)
df.head()

In [ ]:
# Verify imputation worked
preprocessing.missing_value_summary(df).head(10)

In [ ]:
# AQI category distribution
df["aqi_category"].value_counts(normalize=True).round(3)

---
## Task 2c — Statistical and visual analysis

### Univariate — distribution of pollutants

In [ ]:
fig = eda.plot_distributions(df, by_station=False, log_scale=True)
plt.show()

All pollutants are right-skewed on a linear scale; the log-x histograms
show approximately log-normal distributions — a familiar pattern for
atmospheric concentrations driven by multiplicative dispersion processes.

In [ ]:
fig = eda.plot_boxplot_by_station(df, "pm2_5")
plt.show()

Median PM2.5 is markedly higher at the urban stations (Dongsi, Guanyuan)
than at suburban Huairou. Changping sits between, consistent with its
transitional location.

### Bivariate — PM2.5 vs meteorology and other pollutants

In [ ]:
fig = eda.plot_scatter(df, x="temp", y="pm2_5")
plt.show()

fig = eda.plot_scatter(df, x="wspm", y="pm2_5")
plt.show()

fig = eda.plot_scatter(df, x="no2", y="o3")
plt.show()

* **PM2.5 vs temperature** — a U-shape is visible: highest concentrations
  in cold winters (residential heating) and a secondary lift in hot summers.
* **PM2.5 vs wind speed** — strong negative association: high winds
  disperse fine particles. Many extreme PM2.5 readings cluster at near-zero
  wind speeds (stagnant boundary layer).
* **NO₂ vs O₃** — the classic photochemical anti-correlation: ozone forms
  when NO₂ is depleted by sunlight, so the two rarely peak together.

### Multivariate — correlation structure

In [ ]:
fig = eda.plot_correlation_heatmap(df)
plt.show()

Key correlations:
* PM2.5 and PM10 are tightly linked (r ≈ 0.85): coarse and fine particles
  share emission sources and dispersion regimes.
* PM2.5, SO₂, NO₂, CO are positively correlated — all combustion-related.
* O₃ anti-correlates with NO₂ and CO, consistent with photochemical chemistry.
* Wind speed has a weak-to-moderate negative correlation with all primary
  pollutants — dispersion effect.

### Temporal — trends, diurnal cycles, seasonality

In [ ]:
fig = eda.plot_monthly_trend(df, "pm2_5")
plt.show()

In [ ]:
fig = eda.plot_diurnal_cycle(df, "pm2_5")
plt.show()

In [ ]:
fig = eda.plot_seasonal_boxplot(df, "pm2_5")
plt.show()

In [ ]:
fig = eda.plot_aqi_category_distribution(df)
plt.show()

**Temporal patterns**

* **Monthly** — a downward trend across 2013–2017 is visible at all
  stations, consistent with Beijing's emission-control programmes.
  Winter spikes dominate the inter-annual signal.
* **Diurnal** — PM2.5 typically dips in early afternoon (boundary-layer
  growth) and peaks late evening. Urban stations show stronger diurnal
  amplitude.
* **Seasonal** — winter > autumn ≈ spring > summer for PM2.5; the gap
  between urban and suburban stations is widest in winter.
* **AQI categories** — the suburban stations spend a much larger share
  of hours in the *Excellent* / *Good* bands than the urban stations.

---
## Task 3 — Model building

### Problem framing

**Task:** regression — predict the *next hour's* PM2.5 concentration at
the same station, given current pollutant + meteorology readings and
lagged PM2.5 history (1 h, 3 h, 24 h).

**Why this framing?**

* Operationally useful: short-horizon nowcasts feed health alerts and
  exposure estimates.
* Lag-1 PM2.5 is a strong predictor (autocorrelation), giving the model
  a realistic baseline rather than learning from noise.
* Tractable on a laptop — fits within typical assessment compute budgets.

### Modelling decisions

1. **Train/test split** — last 20 % of each station's series held out
   (`time_series_split`). Random shuffling would leak future values into
   training and inflate metrics.
2. **Preprocessor inside the pipeline** — `StandardScaler` for numerics,
   `OneHotEncoder` for `station`, `wd`, `season`. Keeps scaling stats out
   of the test fold.
3. **Two models compared** — `LinearRegression` baseline and
   `RandomForestRegressor` (handles non-linearity and feature interactions).
4. **Metrics** — RMSE (penalises large errors, in µg/m³), MAE
   (interpretable mean error), R² (variance explained).

In [ ]:
results = model.train_and_evaluate(df, tune=False)
print("Baseline (LinearRegression):", results["baseline_metrics"])
print("Random Forest:               ", results["rf_metrics"])

### Feature importances

In [ ]:
results["feature_importances"]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
fi = results["feature_importances"].iloc[::-1]
ax.barh(fi["feature"], fi["importance"], color="#4c72b0")
ax.set_title("Top 15 feature importances — RandomForest PM2.5 model")
ax.set_xlabel("Importance")
fig.tight_layout()
plt.show()

### Predicted vs actual

In [ ]:
preds = results["test_predictions"]

fig, ax = plt.subplots(figsize=(7, 6))
sample = preds.sample(min(len(preds), 4000), random_state=42)
ax.scatter(sample["actual"], sample["predicted"], alpha=0.3, s=10)
lim = max(sample["actual"].max(), sample["predicted"].max())
ax.plot([0, lim], [0, lim], "r--", linewidth=1)
ax.set_xlabel("Actual PM2.5 (µg/m³)")
ax.set_ylabel("Predicted PM2.5 (µg/m³)")
ax.set_title("RandomForest — actual vs predicted (test set)")
fig.tight_layout()
plt.show()

In [ ]:
# A short stretch of actual vs predicted at one station
station_demo = "Dongsi"
sub = preds[preds["station"] == station_demo].sort_values("datetime").iloc[:24*14]  # 2 weeks

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(sub["datetime"], sub["actual"], label="Actual", linewidth=1.2)
ax.plot(sub["datetime"], sub["predicted"], label="Predicted",
        linewidth=1.2, alpha=0.8)
ax.set_title(f"Predicted vs actual PM2.5 — {station_demo} (first 2 test weeks)")
ax.set_ylabel("PM2.5 (µg/m³)")
ax.legend()
fig.tight_layout()
plt.show()

### Model interpretation

* The Random Forest substantially beats the linear baseline on RMSE,
  MAE, and R² — confirming the value of capturing non-linearities and
  feature interactions.
* `pm2_5_lag1` dominates the importance ranking: PM2.5 is highly
  autocorrelated hour-to-hour. CO and NO₂ (combustion proxies) and
  the longer lags follow.
* The actual-vs-predicted scatter clusters tightly around the 1:1 line
  for moderate concentrations, with more dispersion at extreme values —
  a known difficulty for tree models, which struggle to extrapolate
  beyond their training range.

---
## Summary

* **Tasks 1–3** complete: dataset assembled from four stations, fully
  preprocessed, explored across multiple analytical axes, and used to
  train an evaluated PM2.5 nowcast model.
* The trained model is saved to `models/pm25_rf.joblib` for use by the
  Streamlit application (Task 4).
* See `app/streamlit_app.py` for the interactive dashboard and the
  GitHub repository for version-control evidence (Task 5).